In [1]:
from math import gcd
from functools import reduce
import os
from collections import defaultdict
from itertools import permutations
from contextlib import ExitStack
from enum import Enum

class Transformation(Enum): 
    #the class that defines the transformations that we apply to our series
    P = "P"
    I = "I"
    R = "R"
    RI = "RI"
    def apply(self, series:list[int], t:int, n:int)->list[int]:
        if   self == Transformation.P:  return [(x+t) % n for x in series]
        elif self == Transformation.I:  return [(-x+t) % n for x in series]
        elif self == Transformation.R:  return [(x+t) % n for x in reversed(series)]
        elif self == Transformation.RI: return [(-x+t) % n for x in reversed(series)]

def proliferatingPermutation(transformation:Transformation, series:list[int], t:int, n:int)->list[int]:
    #calculates the permutations obtained from series by appliyng transformation and transposition t.
    #The permutation is stored as a list[int] so that permutations[i] yields the note in the transformed
    #series that is in the same place that i was in the original series
    transformed = transformation.apply(series, t, n)
    index = {value: i for (i, value) in enumerate(series)}
    return [transformed[index[value]] for value in range(n)]

def lcm_list(series:list[int])->int:
    return reduce(lambda a,b: a*b//gcd(a,b), series)

def cycleStructure(permutation:list[int],n:int)->(int,list[int]):
    #calculates the lengths of the cycles of the permutation. Returns the order of the permutation and
    #a list with the lengths of the cycles
    visited = [False]*n
    cycles = []
    for i in range(n):
        if not visited[i]:
            current=permutation[i]
            length=1
            while current!=i:
                visited[current]=True
                current=permutation[current]
                length+=1
            cycles.append(length)
    return lcm_list(cycles), sorted(cycles)

def print_justified(series: list[int]) -> None:
    width = max(len(str(x)) for x in series)
    print("["+", ".join(f"{x:{width}d}" for x in series)+"]")

def proliferations(transformation:Transformation, series:list[int], t:int)->None:
    #first major program: takes a transformation with a transposition t and a series. Prints the original
    #and all its proliferations. Also prints the order and the structure of the permutation obtained
    print("Proliferations of", series, "using", transformation.name,"and a transposition of", t, "tone-fractions:")
    print_justified(series)
    n=len(series)
    permutation = proliferatingPermutation(transformation, series, t, n)
    current = [permutation[x] for x in series]
    while current != series:
        print_justified(current)
        current = [permutation[x] for x in current]
    (order,cycles)=cycleStructure(permutation,n) 
    print("Order:", order)
    print("Structure:", cycles)

def proliferations_data(transformation:Transformation, t:int, n:int, path:str)->None:
    #second major program: takes a transformation, a transposition t, a number of notes n and a path.
    #The program creates in this path a folder for the transformation, another folder inside for the
    #number of notes and 3 more subfolders inside it, called "CompleteList", "Data_Orders" and 
    #"Data_Structures". Finally, a txt file inside each of them distinguishing the transposition t.
    #The file in "CompleteList" shows every possible series starting with 0 and the number of proliferations 
    #that it produces. The file in "Data_Orders" shows how many series generate permutations of each possible
    #order, and the file in "Data_Structures" shows how many saries generate permutations of each possible 
    #structure. 
    base_path = os.path.join(path, transformation.name, f"Proliferations_{n}_notes")
    dirs = {
        "Data_Structures": os.path.join(base_path, "Data_Structures"),
        "CompleteList": os.path.join(base_path, "CompleteList"),   #<------------ comment if you don't want the CompleteList
        "Data_Orders": os.path.join(base_path, "Data_Orders")
    }
    for d in dirs.values():
        os.makedirs(d, exist_ok=True)
    file_paths = {key: os.path.join(dirs[key], f"transposition{t}.txt") for key in dirs}
    occurrence_orders = defaultdict(int)
    occurrence_structures = defaultdict(int)
    with ExitStack() as stack:
        files = {key: stack.enter_context(open(file_paths[key], "w")) for key in file_paths}
        for seriesWithout0 in permutations(range(1, n)):
            completeSeries = (0,) + (seriesWithout0)
            permutation = proliferatingPermutation(transformation, completeSeries, t, n)
            order, structure = cycleStructure(permutation, n)
            files["CompleteList"].write(f"{completeSeries} --> {order}\n")  #<------------ comment if you don't want the CompleteList
            occurrence_orders[order] += n
            occurrence_structures[(order, tuple(structure))] += n
        for k, v in sorted(occurrence_orders.items()):
            files["Data_Orders"].write(f"{k}: {v}\n")
        for k, v in sorted(occurrence_structures.items()):
            files["Data_Structures"].write(f"{k}: {v}\n")

In [2]:
#Examples used in article:

#Stockhausen in freundschtaft: C F E C# B Bb Eb Ab D G A F#
series1=[0, 5, 4, 1, 11, 10, 3, 8, 2, 7, 9, 6]
    
#Op. 31 schoenberg:  Bb E F# D# F A D C# G G# B C
series2=[0, 6, 8, 5, 7, 11, 4, 3, 9, 10, 1, 2]
    
#Op. 24 Webern: B Bb D  Eb G F# G# E F C C# A
series3=[0, 11,  3, 4, 8, 7, 9, 5, 6, 1, 2, 10]
    
#Denisov sonata: B C A G# A# F F# G E D# C# D
series4=[0, 1, 10, 9, 11, 6, 7, 8, 5, 4, 2, 3]
    
#Symphony Webern: A F# G Ab E F B Bb D C# C Eb
series5=[0, 9, 10, 11, 7, 8, 2, 1, 5, 4, 3, 6]
    
#Acento en rosa: G C F# G# D# E A C# D F B A#
series6=[0, 5, 11, 1, 8, 9, 2, 6, 7, 10, 4, 3]
    
#Au dela du hazard: C G\# G C\# E D A\# D\# B F F\# A
series7=[0, 8, 7, 1,  4, 2, 10, 3, 11, 5, 6, 9]


In [3]:
proliferations(Transformation.I, [0,9,10,11,7,8,2,1,5,4,3,6], 2)

Proliferations of [0, 9, 10, 11, 7, 8, 2, 1, 5, 4, 3, 6] using I and a transposition of 2 tone-fractions:
[ 0,  9, 10, 11,  7,  8,  2,  1,  5,  4,  3,  6]
[ 2,  5,  4,  3,  7,  6,  0,  1,  9, 10, 11,  8]
Order: 2
Structure: [1, 1, 2, 2, 2, 2, 2]


In [4]:
proliferations(Transformation.P,series1,1)

Proliferations of [0, 5, 4, 1, 11, 10, 3, 8, 2, 7, 9, 6] using P and a transposition of 1 tone-fractions:
[ 0,  5,  4,  1, 11, 10,  3,  8,  2,  7,  9,  6]
[ 1,  6,  5,  2,  0, 11,  4,  9,  3,  8, 10,  7]
[ 2,  7,  6,  3,  1,  0,  5, 10,  4,  9, 11,  8]
[ 3,  8,  7,  4,  2,  1,  6, 11,  5, 10,  0,  9]
[ 4,  9,  8,  5,  3,  2,  7,  0,  6, 11,  1, 10]
[ 5, 10,  9,  6,  4,  3,  8,  1,  7,  0,  2, 11]
[ 6, 11, 10,  7,  5,  4,  9,  2,  8,  1,  3,  0]
[ 7,  0, 11,  8,  6,  5, 10,  3,  9,  2,  4,  1]
[ 8,  1,  0,  9,  7,  6, 11,  4, 10,  3,  5,  2]
[ 9,  2,  1, 10,  8,  7,  0,  5, 11,  4,  6,  3]
[10,  3,  2, 11,  9,  8,  1,  6,  0,  5,  7,  4]
[11,  4,  3,  0, 10,  9,  2,  7,  1,  6,  8,  5]
Order: 12
Structure: [12]


In [5]:
proliferations(Transformation.P,series3,4)

Proliferations of [0, 11, 3, 4, 8, 7, 9, 5, 6, 1, 2, 10] using P and a transposition of 4 tone-fractions:
[ 0, 11,  3,  4,  8,  7,  9,  5,  6,  1,  2, 10]
[ 4,  3,  7,  8,  0, 11,  1,  9, 10,  5,  6,  2]
[ 8,  7, 11,  0,  4,  3,  5,  1,  2,  9, 10,  6]
Order: 3
Structure: [3, 3, 3, 3]


In [6]:
proliferations(Transformation.RI,series7,2)

Proliferations of [0, 8, 7, 1, 4, 2, 10, 3, 11, 5, 6, 9] using RI and a transposition of 2 tone-fractions:
[ 0,  8,  7,  1,  4,  2, 10,  3, 11,  5,  6,  9]
[ 5,  8,  9,  3, 11,  4,  0, 10,  1,  7,  6,  2]
[ 7,  8,  2, 10,  1, 11,  5,  0,  3,  9,  6,  4]
[ 9,  8,  4,  0,  3,  1,  7,  5, 10,  2,  6, 11]
[ 2,  8, 11,  5, 10,  3,  9,  7,  0,  4,  6,  1]
[ 4,  8,  1,  7,  0, 10,  2,  9,  5, 11,  6,  3]
[11,  8,  3,  9,  5,  0,  4,  2,  7,  1,  6, 10]
[ 1,  8, 10,  2,  7,  5, 11,  4,  9,  3,  6,  0]
[ 3,  8,  0,  4,  9,  7,  1, 11,  2, 10,  6,  5]
[10,  8,  5, 11,  2,  9,  3,  1,  4,  0,  6,  7]
Order: 10
Structure: [1, 1, 10]


In [7]:
proliferations(Transformation.RI,series2,5)

Proliferations of [0, 6, 8, 5, 7, 11, 4, 3, 9, 10, 1, 2] using RI and a transposition of 5 tone-fractions:
[ 0,  6,  8,  5,  7, 11,  4,  3,  9, 10,  1,  2]
[ 3,  4,  7,  8,  2,  1,  6, 10,  0,  9, 11,  5]
[10,  6,  2,  7,  5, 11,  4,  9,  3,  0,  1,  8]
[ 9,  4,  5,  2,  8,  1,  6,  0, 10,  3, 11,  7]
Order: 4
Structure: [2, 2, 4, 4]


In [8]:
proliferations(Transformation.RI,[0,5,3,2,6,1,4],0)

Proliferations of [0, 5, 3, 2, 6, 1, 4] using RI and a transposition of 0 tone-fractions:
[0, 5, 3, 2, 6, 1, 4]
[3, 6, 1, 5, 4, 2, 0]
[1, 4, 2, 6, 0, 5, 3]
[2, 0, 5, 4, 3, 6, 1]
[5, 3, 6, 0, 1, 4, 2]
[6, 1, 4, 3, 2, 0, 5]
[4, 2, 0, 1, 5, 3, 6]
Order: 7
Structure: [7]


In [9]:
proliferations(Transformation.R,series1,1)

Proliferations of [0, 5, 4, 1, 11, 10, 3, 8, 2, 7, 9, 6] using R and a transposition of 1 tone-fractions:
[ 0,  5,  4,  1, 11, 10,  3,  8,  2,  7,  9,  6]
[ 7, 10,  8,  3,  9,  4, 11,  0,  2,  5,  6,  1]
[ 5,  4,  0, 11,  6,  8,  9,  7,  2, 10,  1,  3]
[10,  8,  7,  9,  1,  0,  6,  5,  2,  4,  3, 11]
[ 4,  0,  5,  6,  3,  7,  1, 10,  2,  8, 11,  9]
[ 8,  7, 10,  1, 11,  5,  3,  4,  2,  0,  9,  6]
[ 0,  5,  4,  3,  9, 10, 11,  8,  2,  7,  6,  1]
[ 7, 10,  8, 11,  6,  4,  9,  0,  2,  5,  1,  3]
[ 5,  4,  0,  9,  1,  8,  6,  7,  2, 10,  3, 11]
[10,  8,  7,  6,  3,  0,  1,  5,  2,  4, 11,  9]
[ 4,  0,  5,  1, 11,  7,  3, 10,  2,  8,  9,  6]
[ 8,  7, 10,  3,  9,  5, 11,  4,  2,  0,  6,  1]
[ 0,  5,  4, 11,  6, 10,  9,  8,  2,  7,  1,  3]
[ 7, 10,  8,  9,  1,  4,  6,  0,  2,  5,  3, 11]
[ 5,  4,  0,  6,  3,  8,  1,  7,  2, 10, 11,  9]
[10,  8,  7,  1, 11,  0,  3,  5,  2,  4,  9,  6]
[ 4,  0,  5,  3,  9,  7, 11, 10,  2,  8,  6,  1]
[ 8,  7, 10, 11,  6,  5,  9,  4,  2,  0,  1,  3]
[ 0,  5,  4,

In [10]:
proliferations(Transformation.R,series4,2)

Proliferations of [0, 1, 10, 9, 11, 6, 7, 8, 5, 4, 2, 3] using R and a transposition of 2 tone-fractions:
[ 0,  1, 10,  9, 11,  6,  7,  8,  5,  4,  2,  3]
[ 5,  4,  6,  7, 10,  9,  8,  1, 11,  0,  3,  2]
[11,  0,  9,  8,  6,  7,  1,  4, 10,  5,  2,  3]
[10,  5,  7,  1,  9,  8,  4,  0,  6, 11,  3,  2]
[ 6, 11,  8,  4,  7,  1,  0,  5,  9, 10,  2,  3]
[ 9, 10,  1,  0,  8,  4,  5, 11,  7,  6,  3,  2]
[ 7,  6,  4,  5,  1,  0, 11, 10,  8,  9,  2,  3]
[ 8,  9,  0, 11,  4,  5, 10,  6,  1,  7,  3,  2]
[ 1,  7,  5, 10,  0, 11,  6,  9,  4,  8,  2,  3]
[ 4,  8, 11,  6,  5, 10,  9,  7,  0,  1,  3,  2]
Order: 10
Structure: [2, 10]


In [ ]:
#Running proliferations_data in a loop for each traqnsformation, n in range(1,max+1) and t in range(1,n) 
#gives all this data of proliferating series up to the desired number of notes max.
#Don't forget to change "desiredFolder" to the actual folder you want the files in.

#Currently it will run for all series up to 12 notes, which is quite slow, at least several hours running.

#The completeList folders are occupy several GB for n=12, since they show every possible series, so if you don't want them, just 
#comment (put a # in the beggining of the line) the lines that are highlighted with an arrow <------------
#The folder Proliferations_data in this repository has the files that running this program creates, including the cases of
#P and I, and excluding the CompleteList folders.

for n in range(1,12+1):
    for t in range(n):
        proliferations_data(Transformation.RI,t,n,r"C:\Users\Pablo\OneDrive\Documentos\PruebaProliferaciones6")
        proliferations_data(Transformation.R ,t,n,r"C:\Users\Pablo\OneDrive\Documentos\PruebaProliferaciones6")
        #proliferations_data(Transformation.P ,t,n,r"C:\Users\Pablo\OneDrive\Documentos\PruebaProliferaciones6")
        #proliferations_data(Transformation.I ,t,n,r"C:\Users\Pablo\OneDrive\Documentos\PruebaProliferaciones6")
        #Running it for P and I will give us useless information, since the permutations are essentially independent of the specific series